In [1]:
from __future__ import annotations

# Python 표준 라이브러리
import re
import shutil
import warnings

from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

# 환경변수
from dotenv import load_dotenv

# LangChain - Document / PDF
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader

# LangChain - Vector DB
from langchain_chroma import Chroma

# LangChain - Embedding
from langchain_openai import OpenAIEmbeddings

# LangChain - LLM
from langchain_openai import ChatOpenAI

# 경고 메시지 숨기기
warnings.filterwarnings("ignore")

# .env 파일 로드
load_dotenv()

C:\Users\Admin\AppData\Local\Temp\ipykernel_14240\415338402.py:17: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


True

In [2]:
import os

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

In [3]:
load_dotenv()

# 현재 Notebook이 rag/ 폴더에서 실행된다고 가정
BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR / "data"

# Chroma DB 저장 위치
VECTOR_DB_DIR = Path.cwd() / "chroma_db"

# 모델 설정
EMBEDDING_MODEL = "text-embedding-3-small"
LLM_MODEL = os.getenv("OPENAI_MODEL")

print("BASE_DIR :", BASE_DIR)
print("DATA_DIR :", DATA_DIR)
print("VECTOR_DB_DIR :", VECTOR_DB_DIR)
print("LLM_MODEL :", LLM_MODEL)

BASE_DIR : c:\Users\Admin\Desktop\LangChain\04. Week_5_Project
DATA_DIR : c:\Users\Admin\Desktop\LangChain\04. Week_5_Project\data
VECTOR_DB_DIR : c:\Users\Admin\Desktop\LangChain\04. Week_5_Project\rag\chroma_db
LLM_MODEL : gpt-4o


In [4]:
pdf_files = sorted(DATA_DIR.glob("*.pdf"))

print(f"PDF 파일 수: {len(pdf_files)}")

for pdf in pdf_files:
    print("-", pdf.name)

PDF 파일 수: 4
- 개인정보 보호법(법률)(제20897호)(20251002).pdf
- 근로기준법(법률)(제20520호)(20251023).pdf
- 소득세법(법률)(제21221호)(20260701).pdf
- 주택임대차보호법(법률)(제21065호)(20260102).pdf


In [5]:
def get_law_name(pdf_path: Path) -> str:
    """
    파일명에서 법률명을 추출한다.
    예:
    근로기준법(법률...pdf → 근로기준법
    """
    return pdf_path.stem.split("(")[0].strip()


def load_pdf(pdf_path: Path) -> list[Document]:
    loader = PyPDFLoader(str(pdf_path))
    documents = loader.load()

    law_name = get_law_name(pdf_path)

    for doc in documents:
        doc.metadata["law_name"] = law_name
        doc.metadata["file_name"] = pdf_path.name

    return documents

In [6]:
raw_documents = []

for pdf_path in pdf_files:
    docs = load_pdf(pdf_path)
    raw_documents.extend(docs)

    print(
        f"{get_law_name(pdf_path)}: "
        f"{len(docs)} pages"
    )

print()
print("전체 Document 수:", len(raw_documents))

개인정보 보호법: 51 pages
근로기준법: 27 pages
소득세법: 134 pages
주택임대차보호법: 11 pages

전체 Document 수: 223


In [7]:
doc = raw_documents[0]

print(doc.metadata)
print("-" * 80)
print(doc.page_content[:2000])

{'producer': 'iText 2.1.7 by 1T3XT', 'creator': 'PyPDF', 'creationdate': '2026-08-06T17:45:07+09:00', 'moddate': '2026-08-06T17:45:07+09:00', 'source': 'c:\\Users\\Admin\\Desktop\\LangChain\\04. Week_5_Project\\data\\개인정보 보호법(법률)(제20897호)(20251002).pdf', 'total_pages': 51, 'page': 0, 'page_label': '1', 'law_name': '개인정보 보호법', 'file_name': '개인정보 보호법(법률)(제20897호)(20251002).pdf'}
--------------------------------------------------------------------------------
법제처                                                            1                                                       국가법령정보센터
개인정보 보호법
 
개인정보 보호법
[시행 2025. 10. 2.] [법률 제20897호, 2025. 4. 1., 일부개정]
개인정보보호위원회 (심사총괄담당관 - 일반 법령해석) 02-2100-3043
개인정보보호위원회 (국제협력담당관 - 국외이전) 02-2100-2484, 2499
개인정보보호위원회 (개인정보보호정책과 - 법령 제ㆍ개정, 아동ㆍ청소년) 02-2100-3057, 3053
개인정보보호위원회 (신기술개인정보과 - 영상정보, 안전조치) 02-2100-3064, 3028
개인정보보호위원회 (데이터안전정책과 - 가명정보, 개인정보안심구역) 02-2100-3088, 3074, 3058, 3079
개인정보보호위원회 (자율보호정책과 - 보호책임자, 자율규제, 보호수준 평가, 처리방침, 영향평가) 02-2100-3083, 30

In [8]:
for pdf_path in pdf_files:
    docs = load_pdf(pdf_path)

    print("=" * 100)
    print(get_law_name(pdf_path))
    print("=" * 100)

    print(docs[0].page_content[:1000])
    print()

개인정보 보호법
법제처                                                            1                                                       국가법령정보센터
개인정보 보호법
 
개인정보 보호법
[시행 2025. 10. 2.] [법률 제20897호, 2025. 4. 1., 일부개정]
개인정보보호위원회 (심사총괄담당관 - 일반 법령해석) 02-2100-3043
개인정보보호위원회 (국제협력담당관 - 국외이전) 02-2100-2484, 2499
개인정보보호위원회 (개인정보보호정책과 - 법령 제ㆍ개정, 아동ㆍ청소년) 02-2100-3057, 3053
개인정보보호위원회 (신기술개인정보과 - 영상정보, 안전조치) 02-2100-3064, 3028
개인정보보호위원회 (데이터안전정책과 - 가명정보, 개인정보안심구역) 02-2100-3088, 3074, 3058, 3079
개인정보보호위원회 (자율보호정책과 - 보호책임자, 자율규제, 보호수준 평가, 처리방침, 영향평가) 02-2100-3083, 3089, 3087, 3096,
3048
개인정보보호위원회 (분쟁조정과 - 분쟁조정, 손해배상책임) 1833-6972, 02-2100-3142
개인정보보호위원회 (범정부마이데이터 추진단(전략기획팀 - 전송요구권(마이데이터)) 02-2100-3173
       제1장 총칙
 
제1조(목적) 이 법은 개인정보의 처리 및 보호에 관한 사항을 정함으로써 개인의 자유와 권리를 보호하고, 나아가 개인
의 존엄과 가치를 구현함을 목적으로 한다. <개정 2014. 3. 24.>
 
제2조(정의) 이 법에서 사용하는 용어의 뜻은 다음과 같다. <개정 2014. 3. 24., 2020. 2. 4., 2023. 3. 14.>
1. “개인정보”란 살아 있는 개인에 관한 정보로서 다음 각 목의 어느 하나에 해당하는 정보를 말한다.
가. 성명, 주민등록번호 및 영상 등을 통하여 개인을 알아볼 수 있는 정보
나. 해당 정보만으

In [9]:
def merge_pdf_pages(documents: Iterable[Document]) -> str:
    return "\n".join(
        doc.page_content
        for doc in documents
    )

In [10]:
law_texts = {}

for pdf_path in pdf_files:
    docs = load_pdf(pdf_path)

    law_name = get_law_name(pdf_path)

    law_texts[law_name] = {
        "text": merge_pdf_pages(docs),
        "source": pdf_path.name,
    }

In [11]:
for law_name, data in law_texts.items():
    print(law_name, ":", len(data["text"]), "characters")

개인정보 보호법 : 101169 characters
근로기준법 : 52602 characters
소득세법 : 263447 characters
주택임대차보호법 : 19222 characters


In [12]:
def normalize_text(text: str) -> str:
    # Windows / Linux 줄바꿈 통일
    text = text.replace("\r\n", "\n")
    text = text.replace("\r", "\n")

    # 탭 제거
    text = text.replace("\t", " ")

    # 한 줄 안에서 반복되는 공백 정리
    text = re.sub(r"[ ]{2,}", " ", text)

    # 과도한 빈 줄 정리
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

In [13]:
for law_name in law_texts:
    law_texts[law_name]["text"] = normalize_text(
        law_texts[law_name]["text"]
    )

In [14]:
ARTICLE_HEADER_PATTERN = re.compile(
    r"제\s*\d+\s*조(?:의\s*\d+)?(?:\s*\([^)]+\))?"
)

In [15]:
for law_name, data in law_texts.items():
    matches = ARTICLE_HEADER_PATTERN.findall(data["text"])

    print("=" * 60)
    print(law_name)
    print("찾은 조문 수:", len(matches))
    print(matches[:10])

개인정보 보호법
찾은 조문 수: 731
['제1조(목적)', '제2조(정의)', '제3조(개인정보 보호 원칙)', '제30조', '제4조(정보주체의 권리)', '제5조(국가 등의 책무)', '제4조', '제6조(다른 법률과의 관계)', '제7조(개인정보 보호위원회)', '제2조']
근로기준법
찾은 조문 수: 542
['제63조', '제1조(목적)', '제2조(정의)', '제50조', '제69조', '제139조', '제3조(근로조건의 기준)', '제4조(근로조건의 결정)', '제5조(근로조건의 준수)', '제6조(균등한 처우)']
소득세법
찾은 조문 수: 1732
['제1조(목적)', '제1조', '제2조', '제1조의2(정의)', '제2조', '제2조', '제2조(납세의무)', '제13조', '제\n119조의2', '제1조']
주택임대차보호법
찾은 조문 수: 134
['제1조(목적)', '제2조(적용 범위)', '제3조(대항력 등)', '제2조', '제575조', '제578조', '제536조', '제3조의2(보증금의 회수)', '제3조', '제41조']


In [16]:
def split_articles(
    text: str,
    law_name: str,
    source: str
) -> list[Document]:

    matches = list(
        ARTICLE_HEADER_PATTERN.finditer(text)
    )

    documents = []

    for i, match in enumerate(matches):

        start = match.start()

        if i + 1 < len(matches):
            end = matches[i + 1].start()
        else:
            end = len(text)

        article_text = text[start:end].strip()

        # 제 10 조 → 제10조 형태로 정리
        article_header = match.group()

        article_number_match = re.match(
            r"제\s*(\d+)\s*조(?:의\s*(\d+))?",
            article_header
        )

        if article_number_match:
            main_no = article_number_match.group(1)
            sub_no = article_number_match.group(2)

            if sub_no:
                article_number = f"제{main_no}조의{sub_no}"
            else:
                article_number = f"제{main_no}조"
        else:
            article_number = article_header

        document = Document(
            page_content=article_text,
            metadata={
                "law_name": law_name,
                "article": article_number,
                "source": source,
            }
        )

        documents.append(document)

    return documents

In [17]:
article_documents = []

for law_name, data in law_texts.items():

    docs = split_articles(
        text=data["text"],
        law_name=law_name,
        source=data["source"]
    )

    article_documents.extend(docs)

    print(
        f"{law_name}: "
        f"{len(docs)} articles"
    )

print()
print("전체 조문 수:", len(article_documents))

개인정보 보호법: 731 articles
근로기준법: 542 articles
소득세법: 1732 articles
주택임대차보호법: 134 articles

전체 조문 수: 3139


In [18]:
for doc in article_documents[:5]:
    print("=" * 80)
    print(doc.metadata)
    print("-" * 80)
    print(doc.page_content[:500])
    print()

{'law_name': '개인정보 보호법', 'article': '제1조', 'source': '개인정보 보호법(법률)(제20897호)(20251002).pdf'}
--------------------------------------------------------------------------------
제1조(목적) 이 법은 개인정보의 처리 및 보호에 관한 사항을 정함으로써 개인의 자유와 권리를 보호하고, 나아가 개인
의 존엄과 가치를 구현함을 목적으로 한다. <개정 2014. 3. 24.>

{'law_name': '개인정보 보호법', 'article': '제2조', 'source': '개인정보 보호법(법률)(제20897호)(20251002).pdf'}
--------------------------------------------------------------------------------
제2조(정의) 이 법에서 사용하는 용어의 뜻은 다음과 같다. <개정 2014. 3. 24., 2020. 2. 4., 2023. 3. 14.>
1. “개인정보”란 살아 있는 개인에 관한 정보로서 다음 각 목의 어느 하나에 해당하는 정보를 말한다.
가. 성명, 주민등록번호 및 영상 등을 통하여 개인을 알아볼 수 있는 정보
나. 해당 정보만으로는 특정 개인을 알아볼 수 없더라도 다른 정보와 쉽게 결합하여 알아볼 수 있는 정보. 이 경우
쉽게 결합할 수 있는지 여부는 다른 정보의 입수 가능성 등 개인을 알아보는 데 소요되는 시간, 비용, 기술 등
을 합리적으로 고려하여야 한다.
다. 가목 또는 나목을 제1호의2에 따라 가명처리함으로써 원래의 상태로 복원하기 위한 추가 정보의 사용ㆍ결합
없이는 특정 개인을 알아볼 수 없는 정보(이하 “가명정보”라 한다)
1의2. “가명처리”란 개인정보의 일부를 삭제하거나 일부 또는 전부를 대체하는 등의 방법으로 추가 정보가 없이는
특정 개인을 알아볼 수 없도

{'law_name': '개인정보 보호법', 'article': '제3조', 

In [19]:
chunk_lengths = [
    len(doc.page_content)
    for doc in article_documents
]

print("최소:", min(chunk_lengths))
print("최대:", max(chunk_lengths))
print("평균:", sum(chunk_lengths) / len(chunk_lengths))

최소: 4
최대: 2085
평균: 128.98502707868747


In [20]:
long_documents = sorted(
    article_documents,
    key=lambda x: len(x.page_content),
    reverse=True
)

for doc in long_documents[:10]:
    print(
        doc.metadata["law_name"],
        doc.metadata["article"],
        len(doc.page_content)
    )

개인정보 보호법 제64조 2085
소득세법 제120조 2048
소득세법 제87조 1956
개인정보 보호법 제31조 1769
개인정보 보호법 제29조 1749
소득세법 제60조의4 1591
개인정보 보호법 제28조의8 1470
개인정보 보호법 제2조 1370
소득세법 제13조 1360
소득세법 제13조 1345


In [21]:
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

In [22]:
test_vector = embeddings.embed_query(
    "근로자의 근로시간은 몇 시간인가?"
)

print(type(test_vector))
print(len(test_vector))
print(test_vector[:5])

<class 'list'>
1536
[-0.021453857421875, 0.022186279296875, 0.035186767578125, -0.00884246826171875, -0.0126190185546875]


In [23]:
if VECTOR_DB_DIR.exists():
    shutil.rmtree(VECTOR_DB_DIR)

vectorstore = Chroma.from_documents(
    documents=article_documents,
    embedding=embeddings,
    persist_directory=str(VECTOR_DB_DIR),
    collection_name="legal_rag"
)

In [24]:
print(
    "저장된 Document 수:",
    vectorstore._collection.count()
)

저장된 Document 수: 3139


In [25]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 5
    }
)

In [26]:
query = "근로자의 법정 근로시간은 몇 시간이야?"

results = retriever.invoke(query)

for i, doc in enumerate(results, start=1):

    print("=" * 80)
    print(f"검색 결과 {i}")
    print(
        doc.metadata["law_name"],
        doc.metadata["article"]
    )

    print("-" * 80)
    print(doc.page_content[:800])

검색 결과 1
근로기준법 제50조
--------------------------------------------------------------------------------
제50조의 근로시간을 연장할 수 있다.
법제처 13 국가법령정보센터
근로기준법
② 당사자 간에 합의하면 1주 간에 12시간을 한도로
검색 결과 2
근로기준법 제18조
--------------------------------------------------------------------------------
제18조(단시간근로자의 근로조건) ① 단시간근로자의 근로조건은 그 사업장의 같은 종류의 업무에 종사하는 통상 근로
자의 근로시간을 기준으로 산정한 비율에 따라 결정되어야 한다.
② 제1항에 따라 근로조건을 결정할 때에 기준이 되는 사항이나 그 밖에 필요한 사항은 대통령령으로 정한다.
③ 4주 동안(4주 미만으로 근로하는 경우에는 그 기간)을 평균하여 1주 동안의 소정근로시간이 15시간 미만인 근로
자에 대하여는
검색 결과 3
근로기준법 제52조
--------------------------------------------------------------------------------
제52조제1항의 근로시간을 연장
할 수 있다.<개정 2021. 1. 5.>
③ 상시 30명 미만의 근로자를 사용하는 사용자는 다음 각 호에 대하여 근로자대표와 서면으로 합의한 경우 제1항
또는 제2항에 따라 연장된 근로시간에 더하여 1주 간에 8시간을 초과하지 아니하는 범위에서 근로시간을 연장할
수 있다.<신설 2018. 3. 20.>
1. 제1항 또는 제2항에 따라 연장된 근로시간을 초과할 필요가 있는 사유 및 그 기간
2. 대상 근로자의 범위
④ 사용자는 특별한 사정이 있으면 고용노동부장관의 인가와 근로자의 동의를 받아 제1항과 제2항의 근로시간을
연장할 수 있다. 다만, 사태가 급박하여 고용노동부장관의 인가를 받을 시간이 없는 경우에는 사후에 지체 없이 승
인을 받아야 한다.<개정 2010.

In [28]:
test_queries = [
    "개인정보는 언제 파기해야 하나?",
    "근로자의 법정 근로시간은 몇 시간이야?",
    "거주자의 소득세 납세지는 어디야?",
    "임차인이 대항력을 가지기 위한 조건은 뭐야?"
]

for query in test_queries:

    print("=" * 100)
    print("질문:", query)

    results = retriever.invoke(query)

    for doc in results[:3]:
        print(
            "-",
            doc.metadata["law_name"],
            doc.metadata["article"]
        )

질문: 개인정보는 언제 파기해야 하나?
- 개인정보 보호법 제21조
- 개인정보 보호법 제31조
- 개인정보 보호법 제34조의2
질문: 근로자의 법정 근로시간은 몇 시간이야?
- 근로기준법 제50조
- 근로기준법 제18조
- 근로기준법 제52조
질문: 거주자의 소득세 납세지는 어디야?
- 소득세법 제6조
- 소득세법 제8조
- 소득세법 제114조
질문: 임차인이 대항력을 가지기 위한 조건은 뭐야?
- 주택임대차보호법 제3조
- 주택임대차보호법 제3조
- 주택임대차보호법 제3조의3


In [29]:
llm = ChatOpenAI(
    model=LLM_MODEL,
    temperature=0
)

In [30]:
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
당신은 대한민국 법률 문서를 기반으로 답변하는 법률 정보 도우미입니다.

다음 규칙을 반드시 지키세요.

1. 제공된 법률 문서의 내용만 사용해서 답변하세요.
2. 문서에서 확인할 수 없는 내용은 추측하지 마세요.
3. 답변의 근거가 되는 법률명과 조문을 명시하세요.
4. 여러 법률이 검색되었다면 질문과 직접 관련된 법률을 우선하세요.
5. 사용자가 이해하기 쉽게 설명하되 법률의 의미를 임의로 변경하지 마세요.
6. 근거가 부족하면 "제공된 법률 문서에서 확인하기 어렵습니다."라고 답하세요.

[법률 문서]
{context}
"""
    ),
    (
        "human",
        "{question}"
    )
])

In [31]:
def format_documents(documents: list[Document]) -> str:

    formatted = []

    for doc in documents:

        law_name = doc.metadata.get(
            "law_name",
            "법률명 없음"
        )

        article = doc.metadata.get(
            "article",
            "조문 없음"
        )

        formatted.append(
            f"""
[{law_name} {article}]
{doc.page_content}
""".strip()
        )

    return "\n\n---\n\n".join(formatted)

In [32]:
rag_chain = (
    {
        "context":
            retriever
            | RunnableLambda(format_documents),

        "question":
            RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [33]:
question = "근로자의 법정 근로시간은 몇 시간이야?"

answer = rag_chain.invoke(question)

print(answer)

근로자의 법정 근로시간은 근로기준법 제50조에 따라 1주일에 40시간입니다. 다만, 당사자 간에 합의하면 1주일에 12시간을 한도로 연장할 수 있습니다.
